In [1]:
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import LabelEncoder
from scipy.sparse import csr_matrix
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import zipfile
# import urllib.request
import sys

print("Current Working Directory: ", os.getcwd())

C:\ProgramData\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


Current Working Directory:  C:\Users\Connor\Documents\3. School\Projects\CPTS\Project\code\315SongRecomend


In [2]:
data = pd.read_csv('./data/used/train_triplets.txt', sep='\t', header=None, names=['user_id', 'song_id', 'play_count'])
print(data.head())

                                    user_id             song_id  play_count
0  b80344d063b5ccb3212f76538f3d9e43d87dca9e  SOAKIMP12A8C130995           1
1  b80344d063b5ccb3212f76538f3d9e43d87dca9e  SOAPDEY12A81C210A9           1
2  b80344d063b5ccb3212f76538f3d9e43d87dca9e  SOBBMDR12A8C13253B           2
3  b80344d063b5ccb3212f76538f3d9e43d87dca9e  SOBFNSP12AF72A0E22           1
4  b80344d063b5ccb3212f76538f3d9e43d87dca9e  SOBFOVM12A58A7D494           1


In [ ]:
# test_data = pd.read_csv('./data/used/test_triplets.txt', sep='\t', header=None, names=['user_id', 'song_id', 'play_count'])
# print(test_data.head())

test_data = pd.DataFrame(columns=['user_id', 'song_id', 'play_count'])
# Add songs one by one to test_data, each code line is a the same user and a different songid, with play_count of 1
test_user_id = '0000000000000000000000000000000000000000'
test_song_ids = ['SOZKSUN12A8C13F2C6',  # The Marshall Tucker Band, Can't You See
                 'SOERYLG12A6701F07F', # Jack Johnson,Times Like These
                 'SOIDZZS12A8C137460', # Danny O'Keefe,I'm Sober Now (LP Version)
                 'SOGRSUM12A8C13480A', # Otis Redding,(Sittin' On) The Dock Of The Bay
                 'SOGNKUZ12A6D4FB4DF', # Buffalo Springfield,Kind Woman (LP Version)
                 'SOPUZHS12A6D4F783A'] # The Band,I Shall Be Released (Live) (2001 Digital Remaster)

for song_id in test_song_ids:
    test_data = test_data.append({'user_id': test_user_id, 'song_id': song_id, 'play_count': 1}, ignore_index=True)

# concat test_data to data
data = pd.concat([data, test_data], ignore_index=True)
print(data.head())

   user_id             song_id  play_count
0        0  SOZKSUN12A8C13F2C6          20
1        0  SOERYLG12A6701F07F          20
2        0  SOIDZZS12A8C137460          20
3        0  SOGRSUM12A8C13480A          20
4        0  SOGNKUZ12A6D4FB4DF          20
                                    user_id             song_id  play_count
0  b80344d063b5ccb3212f76538f3d9e43d87dca9e  SOAKIMP12A8C130995           1
1  b80344d063b5ccb3212f76538f3d9e43d87dca9e  SOAPDEY12A81C210A9           1
2  b80344d063b5ccb3212f76538f3d9e43d87dca9e  SOBBMDR12A8C13253B           2
3  b80344d063b5ccb3212f76538f3d9e43d87dca9e  SOBFNSP12AF72A0E22           1
4  b80344d063b5ccb3212f76538f3d9e43d87dca9e  SOBFOVM12A58A7D494           1


In [4]:
# Collaborative filtering
# Encode user_id and song_id
user_encoder = LabelEncoder()
song_encoder = LabelEncoder()

data['user_id'] = user_encoder.fit_transform(data['user_id'].astype(str))
data['song_id'] = song_encoder.fit_transform(data['song_id'].astype(str))

# Create user-item matrix
user_item_matrix = csr_matrix((data['play_count'], (data['user_id'], data['song_id'])))
# Fit KNN model
knn = NearestNeighbors(metric='cosine', algorithm='brute')
knn.fit(user_item_matrix)

,"n_neighbors n_neighbors: int, default=5Number of neighbors to use by default for :meth:`kneighbors` queries.",5
,"radius radius: float, default=1.0Range of parameter space to use by default for :meth:`radius_neighbors`queries.",1.0
,"algorithm algorithm: {'auto', 'ball_tree', 'kd_tree', 'brute'}, default='auto'Algorithm used to compute the nearest neighbors:- 'ball_tree' will use :class:`BallTree`- 'kd_tree' will use :class:`KDTree`- 'brute' will use a brute-force search.- 'auto' will attempt to decide the most appropriate algorithm based on the values passed to :meth:`fit` method.Note: fitting on sparse input will override the setting ofthis parameter, using brute force.",'brute'
,"leaf_size leaf_size: int, default=30Leaf size passed to BallTree or KDTree. This can affect thespeed of the construction and query, as well as the memoryrequired to store the tree. The optimal value depends on thenature of the problem.",30
,"metric metric: str or callable, default='minkowski'Metric to use for distance computation. Default is ""minkowski"", whichresults in the standard Euclidean distance when p = 2. See thedocumentation of `scipy.spatial.distance`_ andthe metrics listed in:class:`~sklearn.metrics.pairwise.distance_metrics` for valid metricvalues.If metric is ""precomputed"", X is assumed to be a distance matrix andmust be square during fit. X may be a :term:`sparse graph`, in whichcase only ""nonzero"" elements may be considered neighbors.If metric is a callable function, it takes two arrays representing 1Dvectors as inputs and must return one value indicating the distancebetween those vectors. This works for Scipy's metrics, but is lessefficient than passing the metric name as a string.",'cosine'
,"p p: float (positive), default=2Parameter for the Minkowski metric fromsklearn.metrics.pairwise.pairwise_distances. When p = 1, this isequivalent to using manhattan_distance (l1), and euclidean_distance(l2) for p = 2. For arbitrary p, minkowski_distance (l_p) is used.",2
,"metric_params metric_params: dict, default=NoneAdditional keyword arguments for the metric function.",None
,"n_jobs n_jobs: int, default=NoneThe number of parallel jobs to run for neighbors search.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",None


In [14]:
# Functions to get recommendations
def get_recommendations(user_id, n_recommendations=10, n_neighbors=20):
    user_id_str = str(user_id)

    # Accept either an original user_id value or an already-encoded user index.
    if user_id_str in set(user_encoder.classes_):
        user_index = user_encoder.transform([user_id_str])[0]
    else:
        try:
            user_index = int(user_id)
        except ValueError as exc:
            raise ValueError(f"Unknown user_id: {user_id}") from exc

    if user_index < 0 or user_index >= user_item_matrix.shape[0]:
        raise ValueError(f"user_index out of range: {user_index}")

    n_neighbors = min(n_neighbors + 1, user_item_matrix.shape[0])
    _, indices = knn.kneighbors(user_item_matrix[user_index], n_neighbors=n_neighbors)

    # Neighbor indices are users; use their listened songs to score recommendations.
    neighbor_user_indices = [idx for idx in indices.flatten() if idx != user_index]
    neighbor_matrix = user_item_matrix[neighbor_user_indices]
    song_scores = neighbor_matrix.sum(axis=0).A1

    # Do not recommend songs the target user already listened to.
    listened_song_indices = user_item_matrix[user_index].indices
    song_scores[listened_song_indices] = -1

    top_song_indices = song_scores.argsort()[::-1][:n_recommendations]
    recommended_song_ids = song_encoder.inverse_transform(top_song_indices)
    return recommended_song_ids

print(get_recommendations('0'))

['SOMARRF12AAF3B483C' 'SOJMBTD12AB017F344' 'SOANOQW12A58A793D2'
 'SOSPXWA12AB0181875' 'SOELHPI12A8C1371E4' 'SOWAROE12A58A7B759'
 'SOKBDHZ12A8C13DF54' 'SOFRQTD12A81C233C0' 'SOYIJIL12A6701F1C1'
 'SOQRYOL12A8C13ECAC']


In [26]:
song_info = pd.read_csv('./data/used/unique_tracks.txt', sep='<SEP>', header=None, names=['track_id', 'song_id', 'artist_name','song_title'])
print(song_info.head())

def id_to_song_info(song_id):
    # Support a single song_id or any sequence returned by the recommender.
    if isinstance(song_id, (list, tuple, np.ndarray, pd.Series, pd.Index)):
        return [id_to_song_info(sid) for sid in list(song_id)]

    song_id_str = str(song_id)
    matches = song_info[song_info['song_id'].astype(str) == song_id_str]
    if matches.empty:
        return 'Unknown Song ID'

    row = matches.iloc[0]
    return f"{row['artist_name']} - {row['song_title']}"

C:\Users\Connor\AppData\Local\Temp\ipykernel_33020\1315558236.py:1: ParserWarning: Falling back to the 'python' engine because the 'c' engine does not support regex separators (separators > 1 char and different from '\s+' are interpreted as regex); you can avoid this warning by specifying engine='python'.
  song_info = pd.read_csv('./data/used/unique_tracks.txt', sep='<SEP>', header=None, names=['track_id', 'song_id', 'artist_name','song_title'])


             track_id             song_id       artist_name         song_title
0  TRMMMYQ128F932D901  SOQMMHC12AB0180CB8  Faster Pussy cat       Silent Night
1  TRMMMKD128F425225D  SOVFVAK12A8C1350D9  Karkkiautomaatti        Tanssi vaan
2  TRMMMRX128F93187D9  SOGTUKN12AB017F4F1    Hudson Mohawke  No One Could Ever
3  TRMMMCH128F425532C  SOBNYVR12A8C13558C       Yerba Brava      Si Vos Querés
4  TRMMMWA128F426B589  SOHSBXH12A8C13B0DF        Der Mystic   Tangle Of Aspens


In [ ]:
# Run recommendations for a user from test_data
# Use original user_id value (string/id), not encoded matrix index.
test_user_id = str(test_data['user_id'].iloc[0])
print("Test user_id:", test_user_id)
# Print each song info on a new line 
for song_id in get_recommendations(test_user_id, n_recommendations=50):
    print(id_to_song_info(song_id))

Test user_id: 0
Eazy E - The Life And Timez Of Eric Wright (Part 3)
Standstill - Cuando
The fFormula - Cold Blooded (Acid Cleanse)
Jack Johnson - Bubble Toes
Nickelback - Slow Motion
Nightwish - Wanderlust
The Receiving End Of Sirens - The Salesman_ The Husband_ The Lover
Harmonia - Sehr kosmisch
Jack Johnson - Posters
Saliva - I Walk Alone
